# Text Summarization 

**Pipeline:** `facebook/bart-large-cnn` → abstractive summary → `vennify/t5-base-grammar-correction`

**Improvements over v1:**
- Reusable `summarize_and_correct()` helper with configurable parameters
- Works with **transformers 5.x** (pipeline task strings removed — uses `AutoModel` directly)
- Batch processing support
- Clean separation of setup, model loading, logic, and demo
- Verbose flag for inspecting intermediate output

## 1. Installation

Run once, then restart the kernel if needed.

In [ ]:
!pip install -q transformers sentencepiece
!pip install -q torch --index-url https://download.pytorch.org/whl/cu126

## 2. Environment Check

In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
label  = f"CUDA — {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "CPU"

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {label}")

PyTorch version : 2.12.0+cpu
Device          : CPU


## 3. Load Models

> **Note — transformers 5.x compatibility:** `pipeline()` removed seq2seq task strings
> (`'summarization'`, `'text2text-generation'`) in version 5.0. We use
> `AutoTokenizer` + `AutoModelForSeq2SeqLM` + `.generate()` directly, which works
> on all versions and gives identical results.

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SUMMARIZER_MODEL = "facebook/bart-large-cnn"
GRAMMAR_MODEL    = "vennify/t5-base-grammar-correction"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading summarizer ...")
sum_tok   = AutoTokenizer.from_pretrained(SUMMARIZER_MODEL)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(SUMMARIZER_MODEL).to(device)

print("Loading grammar corrector ...")
gram_tok   = AutoTokenizer.from_pretrained(GRAMMAR_MODEL)
gram_model = AutoModelForSeq2SeqLM.from_pretrained(GRAMMAR_MODEL).to(device)

print(f"\nBoth models loaded on: {device}")

Loading summarizer ...


[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Loading grammar corrector ...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]


Both models loaded on: cpu


## 4. Summarization Helper

In [ ]:
def summarize_and_correct(
    text: str,
    max_length: int = 130,
    min_length: int = 30,
    num_beams: int = 4,
    length_penalty: float = 2.0,
    verbose: bool = True,
) -> str:
    """Abstractively summarise *text* then grammar-correct the result.

    Args:
        text           : Input document (plain text).
        max_length     : Upper token bound for the generated summary.
        min_length     : Lower token bound for the generated summary.
        num_beams      : Beam-search width — higher = better quality, slower.
        length_penalty : >1 encourages longer summaries; <1 shorter ones.
        verbose        : Print intermediate outputs for inspection.

    Returns:
        Grammar-corrected summary string.
    """
    inputs = sum_tok(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True,
    ).to(device)

    summary_ids = sum_model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=min_length,
        num_beams=num_beams,
        length_penalty=length_penalty,
        early_stopping=True,
    )
    raw_summary = sum_tok.decode(summary_ids[0], skip_special_tokens=True)

    if verbose:
        print("[Stage 1 — Raw BART Summary]")
        print(raw_summary, "\n")
    gram_inputs = gram_tok(
        f"grammar: {raw_summary}",
        return_tensors="pt",
        truncation=True,
    ).to(device)

    gram_ids  = gram_model.generate(gram_inputs["input_ids"], max_length=200)
    corrected = gram_tok.decode(gram_ids[0], skip_special_tokens=True).strip()
    if corrected:
        corrected = corrected[0].upper() + corrected[1:]
    if corrected and corrected[-1] not in ".!?":
        corrected += "."

    return corrected

## 5. Run — Paste Your Paragraph Here

Replace the text inside the triple quotes with any paragraph you want to summarise. Then run the cell.

In [8]:
input_text = input("Enter the text to summarise and correct:\n")

output = summarize_and_correct(
    input_text,
    max_length=130,   
    min_length=30,
    num_beams=4,
    verbose=True,   
)

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(output)

[Stage 1 — Raw BART Summary]
A convolutional neural network (CNN) is a type of feedforward neural network that learns features via filter (or kernel) optimization. CNNs are the de-facto standard in deep learning-based approaches to computer vision[2] and image processing. Some applications of CNNs include: image and video recognition, recommender systems, image segmentation, medical image analysis, natural language processing, brain–computer interfaces and financial time series. 

FINAL SUMMARY
A convolutional neural network (CNN) is a type of feedforward neural network that learns features via filter (or kernel) optimization. CNNs are the de facto standard in deep learning-based approaches to computer vision and image processing.
